In [ ]:
import warnings
import torch
from pathlib import Path

def export_model_to_onnx(model, image_width=512, image_height=512, directory_name="model"):
    base_model_name = directory_name + "/lraspp_mobilenet_v3_large"
    weights_path = Path(base_model_name + ".pt")

    # Paths where ONNX and OpenVINO IR models will be stored.
    onnx_path = weights_path.with_suffix(".onnx")
    if not onnx_path.parent.exists():
        onnx_path.parent.mkdir()
    ir_path = onnx_path.with_suffix(".xml")

    # read state dict, use map_location argument to avoid a situation where weights are saved in cuda (which may not be unavailable on the system)
    state_dict = torch.load(weights_path, map_location="cpu")
    # load state dict to model
    model.load_state_dict(state_dict)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        if not onnx_path.exists():
            dummy_input = torch.randn(1, 3, image_height, image_width)
            torch.onnx.export(
                model,
                dummy_input,
                onnx_path,
            )
            print(f"ONNX model exported to {onnx_path}.")
        else:
            print(f"ONNX model {onnx_path} already exists.")


In [3]:
import openvino.runtime as ov

core = ov.Core()
core.add_extension("/Data_large/marine/PythonProjects/MMDET/runscripts/Deploy/mmdeploy-dep/onnxruntime-linux-x64-1.8.1/lib/libonnxruntime.so")

RuntimeError: Cannot add extension. Cannot find entry point to the extension library